# Bundesliga Team Extractor

This is the first notebook in the team-form workflow. It reads `match_ids_1.json` from the configured SofaScore match-ID directory, validates the fixture structure, and extracts every unique home and away team.

The resulting team dictionary is saved as `bundesliga_teams.json` in the configured project output directory. Run this notebook before `02_extract_team_recent_form.ipynb`, which uses that generated file to retrieve each team's recent form.


In [1]:
# Resolve the project root and import authoritative data locations.
import sys
from pathlib import Path


# Handle project root for reuse in the workflow.
def _locate_project_root() -> Path:
    starts = []
    vscode_notebook = globals().get("__vsc_ipynb_file__")
    if isinstance(vscode_notebook, str) and vscode_notebook.strip():
        starts.append(Path(vscode_notebook).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())

    checked = set()
    # Process each available item while preserving the current workflow state.
    for start in starts:
        # Process each available item while preserving the current workflow state.
        for candidate in (start, *start.parents):
            if candidate in checked:
                continue
            checked.add(candidate)
            if (candidate / "project_paths.py").is_file():
                return candidate
    raise FileNotFoundError(
        "Could not locate project_paths.py. Start Jupyter from the Kickbase "
        "project root or open this notebook from within that project."
    )


# Set workflow configuration value: _PROJECT_ROOT.
_PROJECT_ROOT = _locate_project_root()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from project_paths import (
    SOFASCORE_MATCH_IDS_DIR,
    SOFASCORE_REFERENCE_DIR,
    ensure_directory,
)


In [2]:
# 1. Imports
import json
from pathlib import Path
from typing import Any

# Handle expected failures with a clear, actionable message.
try:
    import pandas as pd
    from IPython.display import display
except ImportError as exc:
    raise ImportError(
        "pandas is required. Install it in this Jupyter kernel, then restart the kernel."
    ) from exc


In [3]:
# 2. Matchday and Project File Paths
MATCHDAY = 1
# Validate the input before continuing with later processing.
if not isinstance(MATCHDAY, int) or isinstance(MATCHDAY, bool) or MATCHDAY < 1:
    raise ValueError("MATCHDAY must be a positive integer.")

input_path = SOFASCORE_MATCH_IDS_DIR / f"match_ids_{MATCHDAY}.json"
output_path = ensure_directory(SOFASCORE_REFERENCE_DIR) / "bundesliga_teams.json"

print(f"Input file: {input_path}")
print(f"Output file: {output_path}")


Input file: C:\kickbase project\outputs\sofascore\match_ids\match_ids_1.json
Output file: C:\kickbase project\outputs\sofascore\reference\bundesliga_teams.json


In [4]:
# 3. Load and Validate match_ids_1.json
try:
    fixtures = json.loads(input_path.read_text(encoding="utf-8"))
except FileNotFoundError as exc:
    raise FileNotFoundError(
        f"Input file not found: {input_path}. Run the SofaScore match-ID notebook for MATCHDAY={MATCHDAY}."
    ) from exc
except UnicodeDecodeError as exc:
    raise ValueError(f"Input file is not valid UTF-8: {input_path}") from exc
except json.JSONDecodeError as exc:
    raise ValueError(
        f"Input file is not valid JSON (line {exc.lineno}, column {exc.colno}): "
        f"{input_path}"
    ) from exc
except OSError as exc:
    raise OSError(f"Could not read input file {input_path}: {exc}") from exc

# Validate the input before continuing with later processing.
if not isinstance(fixtures, list):
    raise ValueError("The input JSON must contain a top-level list of fixtures.")

required_fields = {
    "home_team",
    "home_team_id",
    "away_team",
    "away_team_id",
    "match_id",
}

# Check whether positive integer for reuse in the workflow.
def is_positive_integer(value: Any) -> bool:
    return isinstance(value, int) and not isinstance(value, bool) and value > 0

# Process each available item while preserving the current workflow state.
for fixture_number, fixture in enumerate(fixtures, start=1):
    # Validate the input before continuing with later processing.
    if not isinstance(fixture, dict):
        raise ValueError(f"Fixture {fixture_number} must be a JSON object.")

    missing_fields = required_fields.difference(fixture)
    # Validate the input before continuing with later processing.
    if missing_fields:
        missing_text = ", ".join(sorted(missing_fields))
        raise ValueError(f"Fixture {fixture_number} is missing: {missing_text}.")

    # Validate the input before continuing with later processing.
    if not is_positive_integer(fixture["match_id"]):
        raise ValueError(f"Fixture {fixture_number} has no valid positive match_id.")

    # Process each available item while preserving the current workflow state.
    for side in ("home", "away"):
        team_name = fixture[f"{side}_team"]
        team_id = fixture[f"{side}_team_id"]
        # Validate the input before continuing with later processing.
        if not isinstance(team_name, str) or not team_name.strip():
            raise ValueError(
                f"Fixture {fixture_number} has no valid {side}_team name."
            )
        # Validate the input before continuing with later processing.
        if not is_positive_integer(team_id):
            raise ValueError(
                f"Fixture {fixture_number} has no valid positive {side}_team_id."
            )

print(f"Loaded and validated {len(fixtures)} fixture(s).")


Loaded and validated 9 fixture(s).


In [5]:
# 4. Extract Unique Teams
teams: dict[str, dict[str, Any]] = {}

# Process each available item while preserving the current workflow state.
for fixture_number, fixture in enumerate(fixtures, start=1):
    # Process each available item while preserving the current workflow state.
    for side in ("home", "away"):
        team_id = fixture[f"{side}_team_id"]
        team_name = fixture[f"{side}_team"].strip()
        team_key = str(team_id)

        existing = teams.get(team_key)
        # Validate the input before continuing with later processing.
        if existing is not None and existing["team"] != team_name:
            raise ValueError(
                f"Team ID {team_id} has conflicting names: "
                f"{existing['team']!r} and {team_name!r}."
            )

        teams[team_key] = {
            "team_id": team_id,
            "team": team_name,
        }

print(f"Extracted {len(teams)} unique team(s).")
teams_df = pd.DataFrame(teams.values())
display(teams_df)


Extracted 18 unique team(s).


,team_id,team
0,2672,FC Bayern München
1,2677,VfB Stuttgart
2,2671,1. FC Köln
3,2569,TSG Hoffenheim
4,2547,1. FC Union Berlin
5,2674,Eintracht Frankfurt
6,2556,1. FSV Mainz 05
7,2561,SC Paderborn 07
8,36360,RB Leipzig
9,2527,Borussia M'gladbach


In [6]:
# 5. Save Bundesliga Team File
teams_json = json.dumps(teams, ensure_ascii=False, indent=2)
# Handle expected failures with a clear, actionable message.
try:
    output_path.write_text(teams_json + "\n", encoding="utf-8")
except OSError as exc:
    raise OSError(f"Could not save output file {output_path}: {exc}") from exc

print(f"Saved {len(teams)} teams to {output_path}")
print(teams_json)


Saved 18 teams to C:\kickbase project\outputs\sofascore\reference\bundesliga_teams.json
{
  "2672": {
    "team_id": 2672,
    "team": "FC Bayern München"
  },
  "2677": {
    "team_id": 2677,
    "team": "VfB Stuttgart"
  },
  "2671": {
    "team_id": 2671,
    "team": "1. FC Köln"
  },
  "2569": {
    "team_id": 2569,
    "team": "TSG Hoffenheim"
  },
  "2547": {
    "team_id": 2547,
    "team": "1. FC Union Berlin"
  },
  "2674": {
    "team_id": 2674,
    "team": "Eintracht Frankfurt"
  },
  "2556": {
    "team_id": 2556,
    "team": "1. FSV Mainz 05"
  },
  "2561": {
    "team_id": 2561,
    "team": "SC Paderborn 07"
  },
  "36360": {
    "team_id": 36360,
    "team": "RB Leipzig"
  },
  "2527": {
    "team_id": 2527,
    "team": "Borussia M'gladbach"
  },
  "2598": {
    "team_id": 2598,
    "team": "SV 07 Elversberg"
  },
  "2681": {
    "team_id": 2681,
    "team": "Bayer 04 Leverkusen"
  },
  "2673": {
    "team_id": 2673,
    "team": "Borussia Dortmund"
  },
  "2676": {
    "